In [ ]:
import os
import json
import pickle
import pandas as pd
import time
%load_ext autoreload
%autoreload 2
import gemini_chat_mods

### Thematic analysis notebook modified from "LLM-in-the-loop: Leveraging Large Language Model for Thematic Analysis." Authors: Dai, Shih-Chieh and Xiong, Aiping and Ku, Lun-Wei. EMNLP 2023.

In [ ]:
results_dir = 'codes-DATE'
from pathlib import Path
Path(os.path.join('../results',results_dir)).mkdir(parents=True, exist_ok=True)

Step 1: Generating initial codes. Creates an init_codes.json file with output. Input start line and end line you want from the filtered_output.txt file.

In [ ]:
init_codes_out_file = os.path.join('../results',results_dir,'init_codes.json')
exemplar_file = "exemplars.txt"
init_coding_task_file = os.path.join('../prompts/','init_coding_task.txt')


initial_prompt = gemini_chat_mods.create_exemplar_prompt(exemplar_file, init_coding_task_file)


line_start = 0
chunk_size = 195
max_line = 195
is_multipart = True if max_line > (chunk_size - line_start) else False


while line_start < max_line:
    # use min() to make sure response chunk doesn't go past 796
    line_end = min(line_start + chunk_size, max_line)

    prompt_for_codes, text_for_machine = gemini_chat_mods.load_data("filtered_output.txt", line_start, line_end)
    resps, msgs, content = gemini_chat_mods.init_code_gen(initial_prompt, prompt_for_codes, init_codes_out_file=init_codes_out_file)
    line_start += chunk_size

Step 2: Generates summary of the codes in reponse #: code: definition format. Saves it to summary.json file.

In [ ]:
summary_out_file = os.path.join('../results',results_dir,'summary.json')

# Two options: need to tell combine multiple responses, or just use the past message history
if is_multipart:
    if os.path.exists(init_codes_out_file):
        with open(init_codes_out_file, "r", encoding="utf-8") as f:
            try:
                all_coded_responses = json.load(f)
            except Exception as e:
                print("Couldn't load:", e)
    # print(all_coded_responses)
    # print("\n".join(all_coded_responses))
    msgs = gemini_chat_mods.create_msgs_object_for_coding_summary_from_responses("\n".join(all_coded_responses), exemplar_file)
else:
    msgs = gemini_chat_mods.create_msgs_object_for_coding_summary_from_msg_and_response_history(resps, msgs)

# sum_codes, codes = gemini_chat_mods.sum_code_gen(init_codes, msgs, summary_out_file=summary_out_file)
sum_codes, codes = gemini_chat_mods.sum_code_gen_from_messages(msgs, summary_out_file=summary_out_file)

In [ ]:
# Dump information that may be used later. sum_codes is in summary_out_file
unique_codes_out_file = os.path.join('../results',results_dir,'unique_codes_from_summary.json')
with open(unique_codes_out_file, "w", encoding="utf-8") as f:
        json.dump(codes, f, indent=2, ensure_ascii=False)

Step 3: Grouping codes into themes based on question input. Saves to revised_theme.json

In [ ]:
revised_themes_outfile = os.path.join('../results',results_dir,'revised_themes_and_definitions.json')
question = "What is computer science?"
machine_themes, msgs_theme = gemini_chat_mods.codes_to_themes_and_definitions(sum_codes, codes, question, json_out_file=revised_themes_outfile)

In [ ]:
def backup_discussion(msgs_theme_obj, msgs_theme_outfile, rev_themes_json, rev_themes_outfile):
    with open(msgs_theme_outfile, 'wb') as f:
        pickle.dump(msgs_theme_obj, f)

    with open(rev_themes_outfile, "w", encoding="utf-8") as f:
            json.dump(rev_themes_json, f, indent=2, ensure_ascii=False)

In [ ]:
# Save out copies of msgs_theme and machine themes before revisions so if we need to pick up at this spot in the future, we can.
msgs_theme_outfile = os.path.join('../results',results_dir,'msgs_theme_initial.pkl')
initial_themes_and_definitions_outfile = os.path.join('../results',results_dir,'initial_themes_and_definitions.json')


backup_discussion(msgs_theme, msgs_theme_outfile, machine_themes, initial_themes_and_definitions_outfile)


In [ ]:
backups = True
if backups:
    from pathlib import Path
    Path(os.path.join('../results',results_dir,'backups')).mkdir(parents=True, exist_ok=True)
backup_num=1

In [ ]:
reload_from_backup=True
if reload_from_backup:
    with open(msgs_theme_outfile, "rb") as f:
        msgs_theme = pickle.load(f)

    with open(initial_themes_and_definitions_outfile) as f:
        machine_themes = json.load(f)

In [ ]:
# Add additional codes if neccesary
adding_responses = True
if adding_responses:
    line_start = 196
    chunk_size = 100 
    max_line = 796
    is_multipart = True if max_line > (chunk_size - line_start) else False

    with open(initial_themes_and_definitions_outfile, 'r', encoding='utf-8') as f:
        last_codebook_json = json.load(f)

    
    while line_start < max_line:
        # use min() to make sure response chunk doesn't go past 796
        line_end = min(line_start + chunk_size, max_line)

        prompt_for_codes, text_for_machine = gemini_chat_mods.load_data("filtered_output.txt", line_start, line_end)

        additional_coding_out_file = os.path.join('../results',results_dir,f'additional_coding_new_responses_{line_start}_{line_end}.json')
        new_codebook_out_file = os.path.join('../results',results_dir,f'codebook_new_responses_{line_start}_{line_end}.json')

        resps, msgs_theme, content = gemini_chat_mods.examine_additional_responses_to_revise_codebook(question, prompt_for_codes, machine_themes, additional_coding_out_file=additional_coding_out_file)


        summary_out_file = os.path.join('../results',results_dir,f'summary_{line_start}_{line_end}.json')
        msgs_theme = gemini_chat_mods.create_msgs_object_for_coding_summary_from_msg_and_response_history(resps, msgs_theme)
        sum_codes, codes = gemini_chat_mods.sum_code_gen_from_messages(msgs_theme, summary_out_file=summary_out_file)

        machine_themes, msgs_theme = gemini_chat_mods.codes_to_themes_and_definitions(sum_codes, codes, question, json_out_file=new_codebook_out_file)

        merged_codebook_outfile = os.path.join('../results',results_dir,f'merged_codebook_orig_{line_start}_{line_end}.json')
        resp_theme_json, merge_msgs = gemini_chat_mods.merge_codebooks(last_codebook_json, machine_themes, merged_codebook_outfile)

        # resp_theme_json = gemini_chat_mods.create_new_codebook_from_additional_responses(msgs_theme, new_codebook_out_file)
        print(resp_theme_json)
        last_codebook_json = resp_theme_json
        line_start += chunk_size


In [ ]:
# with open(initial_themes_and_definitions_outfile, 'r', encoding='utf-8') as f:
#     init_codebook_json = json.load(f)

# line_start = 196
# line_end = 296
# codebook_to_merge_file = os.path.join('../results',results_dir,f'codebook_new_responses_{line_start}_{line_end}.json')
# with open(codebook_to_merge_file, 'r', encoding='utf-8') as f:
#     codebook_to_merge_json = json.load(f)

# merged_codebook_outfile = os.path.join('../results',results_dir,f'merged_codebook_orig_{line_start}_{line_end}.json')
# resp_theme_json, merge_msgs = gemini_chat_mods.merge_codebooks(init_codebook_json, codebook_to_merge_json, merged_codebook_outfile)

In [ ]:
with open(revised_themes_outfile, "w", encoding="utf-8") as f:
        json.dump(resp_theme_json, f, ensure_ascii=False, indent=2)


In [ ]:
if adding_responses:    
    msgs_theme_outfile = os.path.join('../results',results_dir,'msgs_theme_initial.pkl')
    with open(msgs_theme_outfile, "rb") as f:
        msgs_theme = pickle.load(f)
    
    with open(merged_codebook_outfile, 'r') as f:
        machine_themes = json.load(f)
    

In [ ]:
gemini_chat_mods.print_pretty(machine_themes)

In [ ]:
# If desired, load from backup
load_from_backup = True
if load_from_backup:
    backup_num = 17
    msgs_theme_outfile = os.path.join('../results',results_dir, 'backups', f'msgs_theme_backup{backup_num}.pkl')
    with open(msgs_theme_outfile, "rb") as f:
        msgs_theme = pickle.load(f)
    
    codebook_backup_file = os.path.join('../results',results_dir, 'backups', f'revised_theme_backup{backup_num}.json')
    with open(codebook_backup_file, 'r') as f:
        machine_themes = json.load(f)
    backup_num += 1
    revised_themes_outfile = os.path.join('../results',results_dir,'revised_themes_and_definitions.json')
    gemini_chat_mods.print_pretty(machine_themes)
    


Step 4: We revise the themes:  changing the theme name, adding/deleting/merging themes, and relocating the code to another theme. You can run this cell as many times as necessary. Example of action description in git repository from research paper.

In [ ]:
#put change you want to make in action description
# action_description = """ I'd like to make sure we keep something about 'Personal Feelings/Opinions'. Is there another theme you'd suggest merging it into so that there isn't only one code in its theme? """
# action_description = """ I agree with you that we should not merge'Personal Feelings/Opinions' and should leave it as is. 'Creation/Building' and 'Creative Expression' seems similar. I'd like to merge them into one concept, 'Creativity and Design' while keeping 'Applications' separate. I think this will be easier to code reliably and allow us to identify responses focused on creativity and design. """
# action_description = """ It looks like you got rid of Personal Feelings/Opinions. I thought we were going to keep it as an important area. """
# action_description = """ The themes  "Core Concepts" and "Programming and Problem Solving" seem like they allude to similar ideas. Let's combine them into one theme, with the codes from both of the original themes, to make our codebook more coherent."""
# action_description = """ It seems like one idea that is missing is a computer automating tasks for humans so they don't have to do them. Let's add a "Task Automation code to cover these ideas."""
# action_description = """ 'Society and Humanity/Social Impact' and 'Ethical Considerations' seem like they have some overlapping parts. I propose combining these into a single category in order to reduce the number of codes. """
# action_description = """ We used to have the idea of ethical considerations. I think adding something about ethics to the Society and Humanity code would strengthen that concept. """
# action_description = """ Problem-Solving and Logic/Algorithms seem like they will be challenging to distinguish in short responses. I think we should merge these to make future coding more reliable. """
# action_description = """ The 'replicate thought processes' part of Logic/Algorithms/Problem-Solving seems like it is about something different than logical problem solving. What do you think about adding a code about replicating thought processes in the Societal Impact and Ethics category, and renaming that categoring to Humans and Society? """
# action_description = """ Now that we've renamed the Humans and Society theme, I wonder if Personal Feelings/Opinions would fit in that theme."""
# action_description = """ Your point about Personal Feelings/Opinions is reasonable. Do you have any other suggestions for the codebook? """
# action_description = """ I'm debating whether the concept of computer science being about efficiency is covered in existing codes. We could add it, potentially providing more coverage, but at the cost of a more complex codebook."""
# action_description = """ Communication seems like it's more about Humans and Society rather than Computer Functionality & Interaction. Could we move it, and then perhaps find somewhere else for How Computers Work? """
# action_description = """ Core Computational Problem Solving no longer seems like a great title for the first theme. Is there a change to the theme name that would better capture the codes in that theme? """
# action_description = """ Let's move How Computers Work to Computational Foundations and Communication to Humans and Society. That removes a theme with only two codes, and How Computers Work is a core aspect of computational foundations. """
# action_description = """ Let's make sure the definition for How Computers Work covers how information is represented in memory by adding a clause after 'understanding how computers function internally' like ' and represent information in memory'. This woud make clear that 'Data' is about storing and working with data and not the low level details. """
# action_description = """ The artificial intelligence theme seems like it's about an application more than Humans and Society. Let's move it to the applications theme to make that connection more clear. """
# action_description = """ I would argue that AI is itself an application, not just that it has applications. """
# action_description = """ Could 'Replicating Thought Processes' be renamed to better encompass all ideas that fall under that code? """
action_description = """ I like the renaming of Replicating Thought Processes and I think the definition is somewhat improved, but 'solve complex tasks' feels too broad. What about dropping this phrase? """
msgs_theme = gemini_chat_mods.discussion_llm_revises(machine_themes, msgs_theme, action_description, json_out_file=revised_themes_outfile)





In [ ]:
resp_text = msgs_theme[-1].parts[0].text
print(resp_text)

Step 4 pt2: Run this like to update revised_theme.json with your changes from discussion section. Helpful to run after each change you make to ensure gemini is making the changes requested. 

In [ ]:
revised_themes_outfile

In [ ]:
machine_themes, msgs_theme = gemini_chat_mods.diss_process(msgs_theme, json_out_file=revised_themes_outfile)
if backups:
    msgs_theme_outfile = os.path.join('../results',results_dir,'backups', f'msgs_theme_backup{backup_num}.pkl')
    rev_theme_outfile = os.path.join('../results',results_dir,'backups', f'revised_theme_backup{backup_num}.json')
    backup_discussion(msgs_theme, msgs_theme_outfile, machine_themes, rev_theme_outfile)
    backup_num += 1


Step 5: If you are happy with the result generate final codebook. Saves codebook to codebook.json

In [ ]:
final_code_book_outfile = os.path.join('../results',results_dir,'final_codebook_with_definitions.json')
final_code_book = gemini_chat_mods.final_codebook_gen(final_code_book_outfile, msgs_theme)
# try:
#     with open("codebook_definitions.json", "r", encoding="utf-8") as file:
#         final_code_book = json.load(file)
#         print(final_code_book)
# except FileNotFoundError:
#     print("Error: codebook.json not found. Please ensure the file exists in the correct directory.")
#     exit()
# except json.JSONDecodeError as e:
#     print(f"Error decoding JSON from codebook.json: {e}")
#     exit()
# except Exception as e:
#     print(f"An unexpected error occurred while reading the codebook: {e}")
#     exit()


In [ ]:
final_code_book_outfile = os.path.join('../results',results_dir,'revised_final_codebook_with_definitions.json')
# final_code_book_outfile = os.path.join('../results',results_dir,'final_codebook_with_definitions.json')

try:
    with open(final_code_book_outfile, "r", encoding="utf-8") as file:
        final_code_book = json.load(file)
        gemini_chat_mods.print_pretty(final_code_book)
except FileNotFoundError:
    print(f"Error: {final_code_book_outfile} not found. Please ensure the file exists in the correct directory.")
    exit()

Code new responses

In [ ]:
wics_responses_outfile = "../data/AllWICSDataWithPermissionsSpellcheckedResponseOnly.csv"
loaded_df = pd.read_csv(wics_responses_outfile)
file_contents = loaded_df['wics_def'].to_list()


In [ ]:
from pathlib import Path
Path(os.path.join('../results',results_dir, 'all_labeled_results')).mkdir(parents=True, exist_ok=True)
num_responses_at_once = 10
total_responses = 796 #len(file_contents)
all_records = []
for i in range(0, total_responses, num_responses_at_once):
    # content_to_label = "".join(file_contents[i:(i + num_responses_at_once)])
    content_list = file_contents[i:(i + num_responses_at_once)]
    labeled_text_outfile = os.path.join('../results',results_dir, 'all_labeled_results',f'labeled_responses_from_codebook_{i}_{i+num_responses_at_once}.json')
    code_result = gemini_chat_mods.label(final_code_book, content_list,labeled_text_outfile)
    all_records.extend(gemini_chat_mods.convert_labeled_responses_to_records(code_result, response_offset=i))
    print("Done")
    time.sleep(4) # avoid hitting the rate limit



In [ ]:
pd.DataFrame.from_records(all_records, index='Response').fillna(0).to_csv(os.path.join('../results',results_dir, 'final_labeled_responses_post_discrep_df.csv'))

In [ ]:
from pathlib import Path
Path(os.path.join('../results',results_dir, 'all_labeled_results')).mkdir(parents=True, exist_ok=True)
num_responses_at_once = 10
specific_start_points = [51, 37, 16, 71, 6] # Generated randomly
all_records = []
for i in specific_start_points:
    i = i*num_responses_at_once
    content_to_label = "".join(file_contents[i:(i + num_responses_at_once)])
    labeled_text_outfile = os.path.join('../results',results_dir, 'all_labeled_results',f'labeled_responses_from_codebook_{i}_{i+num_responses_at_once}.json')
    code_result = gemini_chat_mods.label(final_code_book, content_to_label,labeled_text_outfile)
    all_records.extend(gemini_chat_mods.convert_labeled_responses_to_records(code_result, response_offset=i))
    time.sleep(4) # avoid hitting the rate limit

In [ ]:
pd.DataFrame.from_records(all_records, index='Response').fillna(0).to_csv(os.path.join('../results',results_dir, 'final_labeled_responses_post_discrep_subset_df.csv'))

In [ ]:
num_responses_at_once = 10
total_responses = 795
all_backup_records = []

for i in range(0, total_responses, num_responses_at_once):
    labeled_text_outfile = os.path.join('../results',results_dir, 'all_labeled_results',f'labeled_responses_from_codebook_{i}_{i+num_responses_at_once}.json')
    with open(labeled_text_outfile, "r", encoding="utf-8") as file:
        code_result = json.load(file)
    cur_records = gemini_chat_mods.convert_labeled_responses_to_records(code_result, response_offset=i)
    if len(cur_records) != num_responses_at_once:
        print(labeled_text_outfile)
    all_backup_records.extend(cur_records)

len(all_backup_records)
pd.DataFrame.from_records(all_backup_records, index='Response').fillna(0).to_csv(os.path.join('../results',results_dir, 'final_labeled_responses_post_discrep2_df.csv'))

In [ ]:
# Extract ones for human double-coding
num_responses_at_once = 10
starting_indices = [51, 37, 16, 71, 6] # Generated randomly

content_to_human_label = []
indices = []
for starting_index in starting_indices:
    content_to_human_label.extend([s.strip() for s in file_contents[starting_index*num_responses_at_once:(starting_index*num_responses_at_once + num_responses_at_once)]])
    indices.extend(range(starting_index*num_responses_at_once + 1, starting_index*num_responses_at_once + num_responses_at_once + 1))
print(len(content_to_human_label))
print(len(indices))
content_to_human_label_outfile = os.path.join('../results',results_dir,f'content_to_human_label_post_discr.csv')

pd.DataFrame.from_dict({'response_num' : indices, 'response': content_to_human_label}).to_csv(content_to_human_label_outfile, index=False)




### Iteratively improve the final codebook through discussion

In [ ]:
final_code_book_outfile = os.path.join('../results',results_dir,'final_codebook_with_definitions.json')

try:
    with open(final_code_book_outfile, "r", encoding="utf-8") as file:
        final_code_book = json.load(file)
        print(final_code_book)
except FileNotFoundError:
    print(f"Error: {final_code_book_outfile} not found. Please ensure the file exists in the correct directory.")
    exit()

In [ ]:
question = "What is computer science?"
msgs = gemini_chat_mods.get_initial_messages_for_codebook_discrepancy_discussion(final_code_book, question)

In [ ]:
backups = True
if backups:
    from pathlib import Path
    Path(os.path.join('../results',results_dir,'discrepancy_backups')).mkdir(parents=True, exist_ok=True)
backup_num=1

In [ ]:
# response_with_discrepancy = """ Computer science is about learning how computers operate and ways to tell them how to perform a wide variety of functions. I am not sure entirely but it is not all about coding. """
# discrepancy_description = """ I coded this response as including Communication based on the phrase "ways to tell them how to perform a wide variety of functions" but you did not include this code. I think the definition could be improved to make it clear that it includes humans telling things to computers. """
# discrepancy_description = """ Your point about communication being an interaction and being two-sided makes sense, but in other cases, you coded the response as involving communication even when it was not explicitly two sided. For example, you coded the response 'The study of how to effectively communicate your ideas to a computer to do specific tasks.' as including the code Communication. I thus think the response about telling computers things should be coded as Communication. """
# discrepancy_description = """ In your alternative resolution, you suggest explicitly defining a need for two-way interaction for Communication and coding responses about telling computers things or communicating to computers as Programming/Coding. But respondents might not know that instructing computers how to do things involves programming. I'd like to keep this distinction and only include programming/coding if it's clear that that's what a participant means. While this means Communication is not solely about two-way interactions, we preserve a distinction between responses that explicitly mention programming and those that do not. Since responses are from students with limited experience with CS, this distinction is important. """

# response_with_discrepancy = """ Asking a computer, very politely, to do something for you. """
# discrepancy_description = """ I coded the response as included Task Automation and you did not. There were several others where we similarly disagreed, such as "Computer science is the study of computers -- both software and hardware -- and how to use them to create efficient programs to do what you need." All of these responses where we disagreed seemed to be about controlling a computer, with phrases about getting the computer to "do what you need" or getting computers to "to do what you wish." Could we add a code for controlling a computer? """

# response_with_discrepancy = """ CS is problem solving using algorithms imaginatively.  CS is also being able to understand codes. """
# discrepancy_description = """ I coded the response as including Creativity and Design because of the word "imaginitively". Could we add to the code's definition to include creativity involving algorithms in addition to applications?"""
# discrepancy_description = """ I don't think an additional code is necessary. We do allow for coding responses with multiple codes. I agree we don't want all responses involving algorithms to have the Creativity and Design code, though. """

# response_with_discrepancy = """ I think of CS as having two different faces: theory and application. I think applied CS is about learning languages and becoming an effective programmer, whereas theoretical CS is more about proofs and abstraction (similar to pure math). """
# discrepancy_description = """ You coded this with Personal Feelings/Opinions but I did not. While "I think" is an opinion, I don't understand how this response is different from "I think computer science is a subject that covers ways for humans to interact with computers.  It includes multiple different languages to program different kinds of programs that can perform various tasks." which you did not code with Personal Feelings/Opinions. """

# response_with_discrepancy = """ CS is a method of communicating commands towards a machine in a language that is understood by both the user and the machine.  CS combines problem-solving, critical thinking and logic to allow for this.  <--- this question is so hard and I'm bad at explaining things. """
# discrepancy_description = """ You coded this with Personal Feelings/Opinions but I did not. The phrase "I'm bad at explaining things." is a subjective feeling but doesn't seem to be about CS. But I'm not sure how to resolve this discrepancy since the definition already says the feelings should be "towards computer science". """

response_with_discrepancy = """ Computer science... is learning how to communicate with computers so that we can use them to our advantage. The field encompasses the many faces of understanding computers, from coding to development. """
discrepancy_description = """ I coded this with "Society and Humanity" but you did not. I thought the phrase "we can use them to our advantage" was consistent with "Benefit or effects on individuals, groups, and society". """
msgs = gemini_chat_mods.discuss_discrepancy(msgs, response_with_discrepancy, discrepancy_description)


In [ ]:
resp_text = msgs[-1].parts[0].text
print(resp_text)

In [ ]:
revised_codebook_outfile  = os.path.join('../results',results_dir,'revised_final_codebook_with_definitions.json')
resolution_text = """ Use your alternative resolution: Modify the definition of 'Personal Feelings/Opinions' to explicitly include expressions of feelings, opinions, or attitudes related *to the experience of engaging with or discussing* computer science, even if not directly about the field itself. """
revised_final_codebook, msgs_after_discrepancy = gemini_chat_mods.resolve_discrepancy(msgs, resolution_text, final_code_book, json_out_file=revised_codebook_outfile)

if backups:
    msgs_outfile = os.path.join('../results',results_dir,'discrepancy_backups', f'msgs_backup{backup_num}.pkl')
    rev_codebook_outfile = os.path.join('../results',results_dir,'discrepancy_backups', f'revised_codebook_backup{backup_num}.json')
    gemini_chat_mods.backup_discussion(msgs_after_discrepancy, msgs_outfile, revised_final_codebook, rev_codebook_outfile)
    backup_num += 1

In [ ]:
msgs = msgs_after_discrepancy
final_code_book = revised_final_codebook